In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer, accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV




In [ ]:
import os
from pathlib import Path

project_root = Path(r"C:\Users\SAFA MANSOOR\Desktop\project\Invoice-Intelligence-System")
src_path = project_root / "src"

print("Does 'src' folder exist?", src_path.exists())
print("Files inside 'src':", os.listdir(src_path) if src_path.exists() else "Folder not found")

In [ ]:
import sys
from pathlib import Path

project_root = Path(r"C:\Users\SAFA MANSOOR\Desktop\project\Invoice-Intelligence-System")
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

In [ ]:
from data_preprocessing_invoice_risk import InvoiceDataPreprocessor

In [ ]:
csv_file_path = "processed_invoices1.csv"

preprocessor = InvoiceDataPreprocessor(csv_path=csv_file_path)
df = preprocessor.load_data()

df.head()

In [ ]:
df = preprocessor.apply_labels()
df.head()

In [ ]:
sig_feats, non_sig_feats, test_results = preprocessor.perform_statistical_tests()

print("Significant Features for Modeling:", sig_feats)

In [ ]:
X, y = preprocessor.get_modeling_matrices()

In [ ]:
X_train_scaled, X_test_scaled, y_train, y_test = preprocessor.split_and_scale_data(
    test_size=0.2, 
    random_state=42
)

print("\nReady for Machine Learning!")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"y_train distribution:\n{y_train.value_counts()}")

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(),
}
models

In [ ]:
# train all models
results = {}
print("\nTRAINING STARTED!!!!")

for name, model in models.items():
    print(f"\nTraining started for {name}...")
    
    # 1. FIT the model first using your scaled training data!
    model.fit(X_train_scaled, y_train)
    
    # 2. Predict on the test set
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate Metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store the result
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
    }
    
    print(f'{name} trained')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'ROC AUC : {roc_auc:.4f}')

In [ ]:
model_compare = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'Precision': [results[m]['precision'] for m in results],
    'F1-Score': [results[m]['f1'] for m in results],
    'ROC-AUC': [results[m]['roc_auc'] for m in results],
    'Recall': [results[m]['recall'] for m in results],
})

model_compare

In [ ]:
for name, res in results.items():
    print(f"=" * 50)
    print(f"Classification Report for: {name}")
    print(f"=" * 50)
    
    
    report = classification_report(y_test, res['y_pred'])
    print(report)
    print("\n")

In [ ]:
import matplotlib.pyplot as plt


# 1. Extract feature importances from your Random Forest model
importances = results['Random Forest']['model'].feature_importances_
feature_names = X_train_scaled.columns

# 2. Create a dataframe and sort values for clean plotting
feature_importance_df = pd.DataFrame({
    'Feature': feature_names, 
    'Importance': importances
}).sort_values(by='Importance', ascending=True)


# 4. Display the sorted dataframe table (highest importance at the top)
feature_importance_df.sort_values(by='Importance', ascending=False)

### Feature Selection Rationale: Why We Selected 5 Core Features

Although our initial statistical validation and Random Forest baseline evaluated 7 significant features, we intentionally narrowed the final modeling feature set to **5 core financial and quantity variables**:

* `invoice_quantity`
* `invoice_dollars`
* `Freight`
* `total_item_quantity`
* `total_item_dollars`

#### Key Justifications

1. **Preventing Target Leakage & Enabling Real-Time Detection (Excluding `avg_receiving_delay` — 29.1% Importance)**
   * **Operational Timing:** `avg_receiving_delay` is a post-receiving warehouse metric that is unavailable at the exact moment a vendor submits an invoice for accounts payable approval.
   * **Leakage Mitigation:** Because `avg_receiving_delay > 10` is part of the deterministic labeling logic for `flag_invoice`, retaining it forces the model to rely on post-hoc warehouse delays rather than learning underlying billing anomalies.

2. **Eliminating Low-Signal Noise (Excluding `days_po_to_invoice` — 2.4% Importance)**
   * **Minimal Predictive Value:** `days_po_to_invoice` contributed only `0.0241` (2.4%) to total feature importance, making it the weakest predictor by a wide margin.
   * **Model Parsimony:** Removing low-impact features simplifies the production inference pipeline and reduces noise without sacrificing core predictive power.

3. **Real-World Business Alignment (PO-to-Invoice Reconciliation)**
   * The remaining 5 features represent **pure financial and quantity cross-checks** available directly on Purchase Order and Vendor Invoice documents, allowing automated risk flagging prior to payment approval.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV

# 1. Set your 5 chosen features on the preprocessor and unpack scaled data
preprocessor.significant_features = [
    'invoice_quantity',
    'invoice_dollars',
    'Freight',
    'total_item_quantity',
    'total_item_dollars'
]
X_train_scaled, X_test_scaled, y_train, y_test = preprocessor.split_and_scale_data(
    test_size=0.2, random_state=42
)

# Refit baseline models on the same five features for a fair comparison.
results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_pred_proba),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
    }

# 2. Initialize Random Forest (use n_jobs=1 here so GridSearchCV manages parallelism)
rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced',
    n_jobs=1
)

# 3. Hyperparameter grid (216 combinations x 5 folds = 1,080 fits)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 15, 25, 30],         
    'min_samples_split': [2, 5],            
    'min_samples_leaf': [1, 2],             
    'max_features': ['sqrt', 'log2', 0.8]   
}

# 4. Evaluation metric scorer
scorer = make_scorer(f1_score)

# 5. Setup Grid Search with 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    verbose=1,
    n_jobs=-1
)

# 6. Fit on scaled training data
print("Running GridSearchCV on your scaled data...")
grid_search.fit(X_train_scaled, y_train)

print("\nBest Parameters:", grid_search.best_params_)
print(f"Best Cross-Val F1 Score: {grid_search.best_score_:.4f}")


In [ ]:
print("\nEvaluating Best Model on Test Set...")
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}\n")

print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# 1. Gather results from your baseline models (if stored in `results`) and add your tuned GridSearch model
comparison_data = []

# Add baseline models from your results dictionary
for name, res in results.items():
    comparison_data.append({
        'Model': name,
        'Accuracy': res['accuracy'],
        'Precision': res['precision'],
        'Recall': res['recall'],
        'F1-Score': res['f1'],
        'ROC-AUC': res['roc_auc']
    })

# Add your newly tuned Random Forest model from GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
y_pred_tuned = grid_search.best_estimator_.predict(X_test_scaled)
y_prob_tuned = grid_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]

comparison_data.append({
    'Model': 'Tuned Random Forest (GridSearch)',
    'Accuracy': accuracy_score(y_test, y_pred_tuned),
    'Precision': precision_score(y_test, y_pred_tuned, zero_division=0),
    'Recall': recall_score(y_test, y_pred_tuned, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred_tuned, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_prob_tuned)
})

# 2. Convert into a clean DataFrame and sort by F1-Score (or your preferred metric)
model_compare = pd.DataFrame(comparison_data)
model_compare = model_compare.sort_values(by='F1-Score', ascending=False).reset_index(drop=True)

# Display the comparison table
print("--- Model Comparison Table ---")
display(model_compare)

# 3. Automatically select the best model by F1 score
best_model_name = model_compare.loc[0, 'Model']
print(f"\n🏆 The Best Performing Model is: **{best_model_name}**")

# All candidates use the same five selected features and fitted scaler.
if best_model_name == 'Tuned Random Forest (GridSearch)':
    final_best_model = grid_search.best_estimator_
else:
    final_best_model = results[best_model_name]['model']

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


if best_model_name == 'Tuned Random Forest (GridSearch)':
    y_pred = y_pred_tuned
else:
    y_pred = results[best_model_name]['y_pred']

# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Purples',  
    cbar=False,
    xticklabels=['Normal (0)', 'Risk (1)'],
    yticklabels=['Normal (0)', 'Risk (1)'],
)

plt.xlabel('Predicted Label', fontweight='bold')
plt.ylabel('True Label', fontweight='bold')
plt.title(f'Confusion Matrix - {best_model_name}', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Loop through each model in your results dictionary
for name, res in results.items():
  y_pred_proba = res['y_pred_proba']
  auc_score = res['roc_auc']

  # Compute False Positive Rate and True Positive Rate
  fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

  # Plot ROC curve with model name and its AUC score in the label
  plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.4f})', linewidth=2)

# Plot the diagonal 50% random guess line
plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.5000)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC) Curves', fontweight='bold')
plt.legend(loc='lower right', frameon=True)
plt.grid(alpha=0.3)

plt.tight_layout()
# plt.savefig('../reports/roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import joblib

model_path = '../models/invoice_risk_model.pkl'
scaler_path = '../models/risk_scaler.pkl'

if list(final_best_model.feature_names_in_) != list(preprocessor.significant_features):
    raise ValueError('The selected model features do not match the training feature list.')
if list(preprocessor.scaler.feature_names_in_) != list(preprocessor.significant_features):
    raise ValueError('The scaler features do not match the training feature list.')

joblib.dump(final_best_model, model_path)
joblib.dump(preprocessor.scaler, scaler_path)

print(f'✅ Model Saved')
print(f'Model Path : {model_path}')
print(f'Scaler Path: {scaler_path}')